# PerioVision AI: train the per-tooth bone-loss detector (model C, PDCNN) on a free Colab GPU

`Runtime` → `Change runtime type` → **T4 GPU** → Save, then `Runtime` → **Run all** and allow Google Drive.
About 1-1.5 hours on a free T4. If Colab disconnects, Run all again: training resumes from Drive.

Results go to `MyDrive/PerioVision/export_panoramic/` in this account; download that folder and give it to Claude.

In [ ]:
# 0. Settings (defaults chosen for a free T4 GPU)
# Which models to train in this session. Finished models are skipped anyway, so a short session can do one at a time.
RUN = {"A": False, "B": False, "C": True, "D": False}   # D (tooth-detector fine-tune) can also be trained on a CPU
ARCH        = "convnext_tiny"   # backbone for A and B: resnet18 | resnet50 | efficientnet_b3 | convnext_tiny
SIZE        = (1024, 512)       # panoramic film resized to W x H for A and B
EPOCHS_A, EPOCHS_B = 12, 30
BATCH_A, BATCH_B   = 16, 16
# C: per-tooth bone-loss detector. "yolo11s" at 1024 fits one free session (~1-1.5 h on a T4);
# with more GPU time use "yolo11m.pt", 1280, 150 for the most accuracy.
YOLO_C, IMGSZ_C, EPOCHS_C, PATIENCE_C = "yolo11s.pt", 1024, 60, 15
IMGSZ_D, EPOCHS_D = 1280, 40

In [ ]:
# 1. GPU + Google Drive
import subprocess, os, glob, json, shutil
if not shutil.which("nvidia-smi"):
    raise RuntimeError("NO GPU attached. Runtime > Change runtime type > T4 GPU > Save, then Run all again. "
                       "If Colab says 'Cannot connect to GPU backend', this account's free GPU time is used up "
                       "for today: try again later or use another Google account.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/PerioVision"
EXPORT = f"{ROOT}/export_panoramic"
for d in ("runs", "current", "export_panoramic"):
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)

In [ ]:
# 2. Same Ultralytics version as the PerioVision backend, plus the Google-Drive downloader
!pip -q install "ultralytics==8.4.21" gdown
import ultralytics; ultralytics.checks()

In [ ]:
%%writefile /content/convert_pdcnn_coco.py
"""PDCNN panoramic periodontitis dataset (github.com/PuckBlink/PDCNN, 1,747 films) -> YOLO detect.

The dataset ships COCO exports from the VIA annotator: `via_export_coco_BL.json` (per-tooth boxes labelled for
periodontal bone loss) and `via_export_coco_FI.json` (furcation involvement). This converts ONE of them
(--json) into YOLO boxes, keeping the categories found in the file (printed with their counts so the
meaning of each class can be checked), and splits films 80 / 10 / 10 (seeded) into train / val / test.

Usage:  python convert_pdcnn_coco.py --json via_export_coco_BL.json --images <folder with the films> --out <folder>
"""
import argparse
import collections
import glob
import json
import os
import random
import shutil

import cv2


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--json", required=True)
    ap.add_argument("--images", required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    coco = json.load(open(args.json, encoding="utf-8"))
    cats = sorted(coco.get("categories", []), key=lambda c: c["id"])
    if not cats:   # VIA sometimes omits categories: fall back to the ids used
        cats = [{"id": i, "name": f"class_{i}"} for i in sorted({a["category_id"] for a in coco["annotations"]})]
    index = {c["id"]: i for i, c in enumerate(cats)}
    print("categories:", [(c["id"], c["name"]) for c in cats])
    print("boxes per category:", collections.Counter(a["category_id"] for a in coco["annotations"]))
    files = {os.path.basename(p).lower(): p for p in glob.glob(os.path.join(args.images, "**", "*"), recursive=True)
             if p.lower().endswith((".jpg", ".jpeg", ".png"))}
    by_image = collections.defaultdict(list)
    for a in coco["annotations"]:
        by_image[a["image_id"]].append(a)
    images = [im for im in coco["images"] if os.path.basename(im["file_name"]).lower() in files and by_image[im["id"]]]
    random.Random(0).shuffle(images)  # audit-ok: seeded film-level split
    n, counts = len(images), collections.Counter()
    for i, im in enumerate(images):
        split = "train" if i < 0.8 * n else "val" if i < 0.9 * n else "test"
        src = files[os.path.basename(im["file_name"]).lower()]
        w, h = im.get("width"), im.get("height")
        if not w or not h:
            h, w = cv2.imread(src, cv2.IMREAD_GRAYSCALE).shape[:2]
        lines = []
        for a in by_image[im["id"]]:
            x, y, bw, bh = a["bbox"]
            if bw <= 1 or bh <= 1:
                continue
            lines.append(f"{index[a['category_id']]} {(x + bw / 2) / w:.6f} {(y + bh / 2) / h:.6f} {bw / w:.6f} {bh / h:.6f}")
        if not lines:
            continue
        for sub in ("images", "labels"):
            os.makedirs(os.path.join(args.out, sub, split), exist_ok=True)
        stem = f"pdcnn_{im['id']}"
        shutil.copy(src, os.path.join(args.out, "images", split, stem + os.path.splitext(src)[1]))
        with open(os.path.join(args.out, "labels", split, stem + ".txt"), "w") as f:
            f.write("\n".join(lines) + "\n")
        counts[split] += 1
    with open(os.path.join(args.out, "pdcnn.yaml"), "w") as f:
        f.write(f"path: {os.path.abspath(args.out)}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
                + "".join(f"  {i}: '{c['name']}'\n" for i, c in enumerate(cats)))
    print("films per split:", dict(counts))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
# 3. Download the four datasets (skipped if already present)
import urllib.request, zipfile
D = "/content/data"; os.makedirs(D, exist_ok=True)
def fetch(url, path):
    if not os.path.exists(path):
        print("downloading", url); urllib.request.urlretrieve(url, path)
    return path

need_A = RUN["A"] and not os.path.exists(f"{EXPORT}/panoramic_screen_metrics.json")
need_B = RUN["B"] and not os.path.exists(f"{EXPORT}/panoramic_severity_metrics.json")
need_C = RUN["C"] and not os.path.exists(f"{EXPORT}/pdcnn_boneloss_metrics.json")
need_D = RUN["D"] and not os.path.exists(f"{EXPORT}/aku_detector_finetune_metrics.json")
print("still to train:", [k for k, v in (("A", need_A), ("B", need_B), ("C", need_C), ("D", need_D)) if v])

# MM-OPG (Hugging Face, Apache-2.0) - only needed for A
if need_A:
    os.makedirs(f"{D}/mmopg", exist_ok=True)
    HF = "https://huggingface.co/datasets/jeffrey423/ToothXpert.MM-OPG-Annotations/resolve/main"
    fetch(f"{HF}/MM-OPG-Train-Raw.json", f"{D}/mmopg/mmopg_train.json")
    fetch(f"{HF}/MM-OPG-Test.json", f"{D}/mmopg/mmopg_test.json")
    fetch(f"{HF}/images_resized_public.zip", f"{D}/mmopg/images.zip")

# BRAR (figshare, CC BY 4.0) - only needed for B
if need_B and not os.path.exists(f"{D}/brar/meta_data.csv"):
    zipfile.ZipFile(fetch("https://ndownloader.figshare.com/files/58062268", f"{D}/brar.zip")).extractall(f"{D}/brar")
    inner = glob.glob(f"{D}/brar/**/meta_data.csv", recursive=True)[0]
    if os.path.dirname(inner) != f"{D}/brar":
        for p in os.listdir(os.path.dirname(inner)): shutil.move(os.path.join(os.path.dirname(inner), p), f"{D}/brar")

# Aga Khan OPG (Zenodo, CC BY 4.0) - only needed for D
if need_D and not glob.glob(f"{D}/aku/**/annotations", recursive=True):
    zipfile.ZipFile(fetch("https://zenodo.org/api/records/10538750/files/Niihhaa/Dataset-v1.zip/content", f"{D}/aku.zip")).extractall(f"{D}/aku")

# PDCNN (public Google Drive folder) - only needed for C
if need_C:
    if not glob.glob(f"{D}/pdcnn/**/*.json", recursive=True):
        !gdown --folder "https://drive.google.com/drive/folders/18qUxeRPHPcCQT9o8AgV5400f05Fu3ISW" -O {D}/pdcnn
    for z in glob.glob(f"{D}/pdcnn/**/*.zip", recursive=True):
        if not os.path.isdir(z[:-4]): zipfile.ZipFile(z).extractall(z[:-4])
    print("PDCNN annotation files:", glob.glob(f"{D}/pdcnn/**/*.json", recursive=True))
print(subprocess.run("du -sh " + D + "/*", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 6. Model C: per-tooth bone-loss detector (PDCNN). Resumes from Drive after a disconnect.
from ultralytics import YOLO
if not need_C:
    print("Model C already trained or switched off in RUN - skipping")
else:
    bl = [p for p in glob.glob(f"{D}/pdcnn/**/*.json", recursive=True) if "BL" in os.path.basename(p)]
    assert bl, "PDCNN bone-loss JSON not found - check the printout of step 3"
    if not os.path.exists("/content/pdcnn_yolo/pdcnn.yaml"):
        !python /content/convert_pdcnn_coco.py --json "{bl[0]}" --images {D}/pdcnn --out /content/pdcnn_yolo
    print(open("/content/pdcnn_yolo/pdcnn.yaml").read())
    run_c = f"{ROOT}/runs/pdcnn_bl"
    if os.path.exists(f"{run_c}/weights/last.pt") and not os.path.exists(f"{run_c}/DONE"):
        YOLO(f"{run_c}/weights/last.pt").train(resume=True)
    elif not os.path.exists(f"{run_c}/DONE"):
        YOLO(YOLO_C).train(data="/content/pdcnn_yolo/pdcnn.yaml", imgsz=IMGSZ_C, epochs=EPOCHS_C, patience=PATIENCE_C, batch=-1,
                           project=f"{ROOT}/runs", name="pdcnn_bl", exist_ok=True, seed=0, deterministic=False,
                           fliplr=0.5, mosaic=0.5, close_mosaic=10, cache="ram", save_period=1)
    open(f"{run_c}/DONE", "w").close()

    # Honest test: detection mAP per class + per-tooth label accuracy on the held-out 10 %
    model_c = YOLO(f"{run_c}/weights/best.pt")
    r = model_c.val(data="/content/pdcnn_yolo/pdcnn.yaml", split="test", imgsz=IMGSZ_C, project=f"{ROOT}/runs", name="pdcnn_test", exist_ok=True)
    def iou(a, b):
        x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1); return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)
    right = total = 0
    for img in glob.glob("/content/pdcnn_yolo/images/test/*"):
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        p = model_c(img, imgsz=IMGSZ_C, conf=0.25, verbose=False)[0]
        H, W = p.orig_shape
        preds = [(b.xyxy[0].tolist(), int(b.cls)) for b in p.boxes]
        for line in open(lab):
            c, cx, cy, w, h = map(float, line.split()); g = [(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H]
            best = max(preds, key=lambda q: iou(g, q[0]), default=None)
            total += 1
            if best is not None and iou(g, best[0]) >= 0.5 and best[1] == int(c): right += 1
    names = model_c.names
    mc = {"task": "per-tooth periodontal bone-loss boxes (PDCNN)", "classes": names,
          "test_mAP50": round(float(r.box.map50), 4), "test_mAP50_95": round(float(r.box.map), 4),
          "test_per_class_mAP50": {names[i]: round(float(v), 4) for i, v in enumerate(r.box.all_ap[:, 0])},
          "test_tooth_found_with_correct_label": round(right / max(total, 1), 4), "test_teeth": total}
    json.dump(mc, open(f"{EXPORT}/pdcnn_boneloss_metrics.json", "w"), indent=2)
    shutil.copy(f"{run_c}/weights/best.pt", f"{EXPORT}/pdcnn_boneloss.pt")
    print(json.dumps(mc, indent=2))

In [ ]:
# 8. Summary of everything in the export folder
for f in sorted(glob.glob(f"{EXPORT}/*")):
    print(f"{os.path.basename(f):45s} {os.path.getsize(f)/1e6:8.1f} MB")
for f in sorted(glob.glob(f"{EXPORT}/*metrics.json")):
    print("\n==", os.path.basename(f)); print(open(f).read())